# KNIME di Google Colab: Balancing Dataset Bank Marketing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Raditxt/eksplorasi-knime-orange-n8n/blob/main/colab/02_knime_balancing_di_colab.ipynb)

Notebook ini adalah pasangan dari workflow KNIME [`Unbalance_Dataset.knwf`](../knime/06-unbalanced-dataset/).

Berbeda dengan Orange, **KNIME tidak bisa dijalankan di Colab** karena KNIME adalah aplikasi desktop berbasis Java, bukan library Python. Karena itu, kompatibilitas KNIME dengan Colab diuji dengan dua cara:

1. **Menulis ulang workflow sebagai kode** (pandas + scikit-learn + imbalanced-learn), lalu membandingkan hasilnya dengan KNIME.
2. **Membawa model KNIME ke Colab** melalui format PMML: model Decision Tree yang dilatih di KNIME dijalankan langsung di Colab.

| Node KNIME | Padanan di notebook ini |
|---|---|
| CSV Reader | `pandas.read_csv(sep=";")` |
| Table Partitioner (70%, stratified, seed 42) | `train_test_split(test_size=0.3, stratify=y, random_state=42)` |
| Equal Size Sampling (Exact, seed 42) | `RandomUnderSampler(random_state=42)` |
| Category to Number | `OrdinalEncoder` |
| SMOTE (k = 5, oversample sampai seimbang, seed 42) | `SMOTE(k_neighbors=5, random_state=42)` |
| Decision Tree Learner (Gini, tanpa pruning, min. 2 record per node) | `DecisionTreeClassifier(criterion="gini", min_samples_leaf=2)` |
| Decision Tree Predictor + Scorer | `model.predict` + fungsi `evaluasi()` |

## 1. Persiapan Lingkungan

Colab sudah menyediakan pandas dan scikit-learn, jadi tinggal memasang imbalanced-learn. Kalau dijalankan di laptop, cukup pakai Python apa saja yang sudah berisi pandas, scikit-learn, dan imbalanced-learn (Python bawaan Orange juga bisa).

In [1]:
%pip install -q imbalanced-learn

In [2]:
import re
import warnings
import xml.etree.ElementTree as ET
from urllib.request import urlopen

import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.tree import DecisionTreeClassifier
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE, SMOTENC

warnings.filterwarnings("ignore")
print("scikit-learn", sklearn.__version__)

scikit-learn 1.9.1


## 2. Memuat Dataset

Dataset dan model PMML dibaca langsung dari repositori GitHub, jadi tidak perlu upload manual.

In [3]:
REPO = "https://raw.githubusercontent.com/Raditxt/eksplorasi-knime-orange-n8n/main"
df = pd.read_csv(f"{REPO}/datasets/bank-full.csv", sep=";")

FITUR_7 = ["age", "job", "marital", "education", "balance", "housing", "loan"]
print(df.shape)
df[FITUR_7 + ["y"]].head()

(45211, 17)


,age,job,marital,education,balance,housing,loan,y
0,58,management,married,tertiary,2143,yes,no,no
1,44,technician,single,secondary,29,yes,no,no
2,33,entrepreneur,married,secondary,2,yes,yes,no
3,47,blue-collar,married,unknown,1506,yes,no,no
4,33,unknown,single,unknown,1,no,no,no


## 3. Fungsi Bantu

Kolom yang dipakai sama dengan CSV Reader di KNIME: `age, job, marital, education, balance, housing, loan`. Kolom `duration` dibuang karena lama telepon baru diketahui setelah telepon selesai (data leakage).

Catatan soal encoding:
- **KNIME** Decision Tree bisa langsung memakai kolom teks (nominal).
- **scikit-learn** hanya menerima angka, jadi kolom kategorikal diubah dengan **one-hot encoding** untuk skenario tanpa balancing, Equal Size Sampling, dan SMOTE-NC.
- Untuk skenario **SMOTE ala KNIME**, kolom kategorikal diubah dengan **ordinal encoding** (satu angka per kategori), meniru node *Category to Number* sebelum SMOTE.

In [4]:
def evaluasi(y_asli, y_pred, nama, n_latih):
    """Meniru node Scorer di KNIME, kelas positif = yes."""
    y_asli = np.asarray(y_asli); y_pred = np.asarray(y_pred)
    tp = int(((y_pred == "yes") & (y_asli == "yes")).sum())
    fp = int(((y_pred == "yes") & (y_asli == "no")).sum())
    fn = int(((y_pred == "no") & (y_asli == "yes")).sum())
    tn = int(((y_pred == "no") & (y_asli == "no")).sum())
    recall = tp / (tp + fn)
    precision = tp / (tp + fp) if tp + fp else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    akurasi = (tp + tn) / len(y_asli)
    # Cohen's kappa, seperti di Scorer KNIME
    n = len(y_asli)
    p_e = ((tp + fp) * (tp + fn) + (fn + tn) * (fp + tn)) / n**2
    kappa = (akurasi - p_e) / (1 - p_e)
    return {"Skenario": nama, "Data latih": n_latih, "TP": tp, "FP": fp, "FN": fn, "TN": tn,
            "Recall": recall, "Precision": precision, "F1": f1, "Accuracy": akurasi, "Kappa": kappa}


def format_persen(tabel):
    """Tampilkan kolom metrik sebagai persen."""
    hasil = tabel.copy()
    for kolom in hasil.columns:
        nama = kolom[-1] if isinstance(kolom, tuple) else kolom
        if nama in ("Recall", "Precision", "F1", "Accuracy"):
            hasil[kolom] = hasil[kolom].map(lambda v: f"{v:.1%}")
        elif nama == "Kappa":
            hasil[kolom] = hasil[kolom].map(lambda v: f"{v:.3f}")
    return hasil


def pohon():
    # Parameter mengikuti Decision Tree Learner di KNIME
    return DecisionTreeClassifier(criterion="gini", min_samples_leaf=2, random_state=42)

In [5]:
def jalankan_skenario(df, fitur, seed=42):
    X = df[fitur]; y = df["y"]
    X_latih, X_uji, y_latih, y_uji = train_test_split(
        X, y, test_size=0.3, stratify=y, random_state=seed)   # Table Partitioner
    kategori = [k for k in fitur if not pd.api.types.is_numeric_dtype(X[k])]
    numerik = [k for k in fitur if k not in kategori]
    print(f"Data latih: {len(X_latih)} baris, data uji: {len(X_uji)} baris")

    onehot = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), kategori)],
                               remainder="passthrough")
    Xo_latih = onehot.fit_transform(X_latih); Xo_uji = onehot.transform(X_uji)

    hasil = []
    # A. Tanpa balancing
    m = pohon().fit(Xo_latih, y_latih)
    hasil.append(evaluasi(y_uji, m.predict(Xo_uji), "A. Tanpa balancing", len(y_latih)))

    # B. Equal Size Sampling
    Xb, yb = RandomUnderSampler(random_state=seed).fit_resample(Xo_latih, y_latih)
    m = pohon().fit(Xb, yb)
    hasil.append(evaluasi(y_uji, m.predict(Xo_uji), "B. Equal Size Sampling", len(yb)))

    # C. SMOTE ala KNIME: Category to Number -> SMOTE -> Decision Tree
    ordinal = ColumnTransformer([("cat", OrdinalEncoder(), kategori)], remainder="passthrough")
    Xn_latih = ordinal.fit_transform(X_latih); Xn_uji = ordinal.transform(X_uji)
    Xc, yc = SMOTE(k_neighbors=5, random_state=seed).fit_resample(Xn_latih, y_latih)
    m_smote = pohon().fit(Xc, yc)
    hasil.append(evaluasi(y_uji, m_smote.predict(Xn_uji), "C. SMOTE ala KNIME", len(yc)))

    # D. SMOTE-NC: SMOTE khusus data campuran numerik + kategorikal
    idx_kat = [fitur.index(k) for k in kategori]
    Xd, yd = SMOTENC(categorical_features=idx_kat, random_state=seed).fit_resample(X_latih, y_latih)
    m = pohon().fit(onehot.transform(pd.DataFrame(Xd, columns=fitur)), yd)
    hasil.append(evaluasi(y_uji, m.predict(Xo_uji), "D. SMOTE-NC", len(yd)))

    tabel = pd.DataFrame(hasil).set_index("Skenario")
    return tabel, dict(model=m_smote, X_uji_angka=Xn_uji, y_uji=y_uji, n_kat=len(kategori))

## 4. Eksperimen 1: 7 Fitur (sama dengan workflow KNIME)

Seperti di KNIME, balancing **hanya diterapkan ke data latih**, sedangkan data uji tetap pada distribusi aslinya.

In [6]:
hasil_7, info_7 = jalankan_skenario(df, FITUR_7)
format_persen(hasil_7)

Data latih: 31647 baris, data uji: 13564 baris


,Data latih,TP,FP,FN,TN,Recall,Precision,F1,Accuracy,Kappa
Skenario,,,,,,,,,,
A. Tanpa balancing,31647,334,726,1253,11251,21.0%,31.5%,25.2%,85.4%,0.175
B. Equal Size Sampling,7404,889,4288,698,7689,56.0%,17.2%,26.3%,63.2%,0.102
C. SMOTE ala KNIME,55890,292,754,1295,11223,18.4%,27.9%,22.2%,84.9%,0.142
D. SMOTE-NC,55890,645,2483,942,9494,40.6%,20.6%,27.4%,74.7%,0.140


### Perbandingan dengan KNIME

Angka KNIME diambil dari laporan Pekan 3 (Scorer pada data uji 13.564 baris).

In [7]:
knime = pd.DataFrame({
    "Recall":    [0.2647, 0.6219, 0.1619],
    "Precision": [0.3055, 0.1681, 0.1278],
    "F1":        [0.2836, 0.2646, 0.1429],
    "Accuracy":  [0.8436, 0.5955, 0.7726],
}, index=["A. Tanpa balancing", "B. Equal Size Sampling", "C. SMOTE ala KNIME"])

perbandingan = pd.concat({"KNIME": knime,
                          "Colab": hasil_7.loc[knime.index, knime.columns]}, axis=1)
format_persen(perbandingan)

KNIME                            Colab            \
                       Recall Precision     F1 Accuracy Recall Precision   
A. Tanpa balancing      26.5%     30.6%  28.4%    84.4%  21.0%     31.5%   
B. Equal Size Sampling  62.2%     16.8%  26.5%    59.6%  56.0%     17.2%   
C. SMOTE ala KNIME      16.2%     12.8%  14.3%    77.3%  18.4%     27.9%   

                                        
                           F1 Accuracy  
A. Tanpa balancing      25.2%    85.4%  
B. Equal Size Sampling  26.3%    63.2%  
C. SMOTE ala KNIME      22.2%    84.9%

**Hasil perbandingan:**

- **Tanpa balancing dan Equal Size Sampling: polanya sama dengan KNIME.** Angkanya tidak identik (selisih beberapa poin persen), dan itu wajar karena:
  1. Generator angka acak Java (KNIME) dan Python berbeda, jadi walaupun seed-nya sama-sama 42, baris yang masuk data latih dan data uji berbeda.
  2. Algoritma Decision Tree-nya berbeda. KNIME memecah kolom nominal langsung berdasarkan kelompok kategori, sedangkan scikit-learn memecah kolom hasil one-hot satu per satu.
- **Jumlah data latih dan data uji sama persis dengan KNIME** (31.647 dan 13.564 baris), begitu juga jumlah data setelah Equal Size Sampling (7.404) dan SMOTE (55.890).
- **SMOTE: hasil Colab jauh lebih baik daripada KNIME** (Precision 27,9% vs 12,8%, Kappa 0,142 vs 0,014). Selisihnya terlalu besar untuk dijelaskan oleh dua alasan di atas, jadi workflow KNIME-nya diperiksa lebih lanjut di bagian 5.

## 5. Memeriksa Workflow KNIME: Menjalankan Model KNIME di Colab (PMML)

Node *Decision Tree Learner* di KNIME menghasilkan model dalam format **PMML** (Predictive Model Markup Language), yaitu format XML standar untuk model machine learning. File PMML bisa diekspor dari KNIME dengan node *PMML Writer*, lalu dibaca di tools lain.

Model dari workflow `Unbalance_Dataset.knwf` sudah diekspor ke folder [`knime/06-unbalanced-dataset/pmml/`](../knime/06-unbalanced-dataset/pmml/):

| File | Asal di workflow KNIME |
|---|---|
| `model_tanpa_balancing.pmml` | Decision Tree Learner (#6) |
| `model_smote.pmml` | Decision Tree Learner (#8), cabang SMOTE |
| `category_to_number.pmml` | Category to Number (#4), berisi tabel kategori → angka |

Pertama, lihat tipe data yang diharapkan tiap model:

In [8]:
def baca(nama):
    return urlopen(f"{REPO}/knime/06-unbalanced-dataset/pmml/{nama}").read().decode()

pmml_baseline = baca("model_tanpa_balancing.pmml")
pmml_smote = baca("model_smote.pmml")
pmml_cat2num = baca("category_to_number.pmml")

def tipe_kolom(pmml):
    return dict(re.findall(r'<DataField name="(\w+)" optype="(\w+)"', pmml))

pd.DataFrame({"Model tanpa balancing (#6)": tipe_kolom(pmml_baseline),
              "Model SMOTE (#8)": tipe_kolom(pmml_smote)})

,Model tanpa balancing (#6),Model SMOTE (#8)
age,continuous,continuous
job,categorical,continuous
marital,categorical,continuous
education,categorical,continuous
balance,continuous,continuous
housing,categorical,continuous
loan,categorical,continuous
y,categorical,categorical


Model SMOTE (#8) mengharapkan kolom `job`, `marital`, `education`, `housing`, dan `loan` sebagai **angka** (`continuous`), karena dilatih dengan data hasil *Category to Number*.

Tapi di workflow, data uji dikirim **langsung** dari *Table Partitioner* ke *Decision Tree Predictor (#11)*, tanpa lewat *Category to Number*, sehingga kolom-kolom itu masih berupa **teks**:

```
Table Partitioner ── data latih → Category to Number → SMOTE → Decision Tree Learner (#8) ─┐
                 └── data uji (masih teks!) ──────────────────────────────→ Decision Tree Predictor (#11)
```

KNIME tidak menampilkan error, tetapi model tidak bisa membaca kolom kategorikal di data uji. Untuk membuktikannya, model PMML dari KNIME dijalankan di Colab dengan dua cara: data diberikan sebagai **teks** (seperti di workflow) dan data yang sudah di-**encode** dengan tabel dari *Category to Number*.

Fungsi di bawah adalah evaluator sederhana untuk model pohon keputusan PMML. Aturan khusus: jika model mengharapkan angka tetapi menerima teks, perbandingan `<=` dianggap salah, sehingga data selalu berbelok ke cabang `>`.

In [9]:
def muat_pohon(pmml):
    root = ET.fromstring(pmml)
    ns = root.tag.split("}")[0][1:]
    return root.find(f".//{{{ns}}}TreeModel/{{{ns}}}Node"), ns


def cocok(node, baris, ns):
    for el in node:
        tag = el.tag.split("}")[1]
        if tag == "True":
            return True
        if tag == "SimplePredicate":
            nilai = baris[el.get("field")]
            if isinstance(nilai, str):            # model butuh angka, data berupa teks
                return el.get("operator") == "greaterThan"
            batas = float(el.get("value"))
            return nilai <= batas if el.get("operator") == "lessOrEqual" else nilai > batas
        if tag == "SimpleSetPredicate":
            daftar = re.findall(r'"([^"]*)"', el.find(f"{{{ns}}}Array").text)
            return (str(baris[el.get("field")]) in daftar) == (el.get("booleanOperator") == "isIn")
    return False


def prediksi_pmml(pmml, data):
    akar, ns = muat_pohon(pmml)
    hasil = []
    for baris in data.to_dict("records"):
        node = akar
        while True:
            anak = next((k for k in node if k.tag.endswith("Node") and cocok(k, baris, ns)), None)
            if anak is None:
                break
            node = anak
        hasil.append(node.get("score"))
    return np.array(hasil)


# Tabel kategori -> angka dari node Category to Number
peta = {}
for kolom, isi in re.findall(r'<FieldColumnPair field="(\w+)" column="in"/>(.*?)</InlineTable>', pmml_cat2num, re.S):
    peta[kolom] = {a: int(b) for a, b in re.findall(r"<pmml:in>(.*?)</pmml:in>\s*<pmml:out>(\d+)</pmml:out>", isi)}
peta["job"]

{'technician': 0,
 'entrepreneur': 1,
 'blue-collar': 2,
 'management': 3,
 'retired': 4,
 'admin.': 5,
 'services': 6,
 'self-employed': 7,
 'unemployed': 8,
 'housemaid': 9,
 'student': 10,
 'unknown': 11}

Model dijalankan pada **seluruh 45.211 baris** data. Pembagian data latih/uji di KNIME memakai generator acak Java yang tidak bisa direproduksi di Python, jadi baris data uji KNIME tidak bisa dipisahkan di sini. Akibatnya angka untuk data yang di-encode akan **terlalu optimis** (sekitar 70% barisnya adalah data latih model). Yang penting di sini adalah perbandingan antara kedua cara pemberian data.

In [10]:
data = df[FITUR_7]
data_angka = data.copy()
for kolom, mapping in peta.items():
    data_angka[kolom] = data_angka[kolom].map(mapping)

uji_pmml = pd.DataFrame([
    evaluasi(df["y"], prediksi_pmml(pmml_smote, data), "Model SMOTE KNIME, data teks (seperti di workflow)", "-"),
    evaluasi(df["y"], prediksi_pmml(pmml_smote, data_angka), "Model SMOTE KNIME, data di-encode", "-"),
]).set_index("Skenario")[["Recall", "Precision", "F1", "Kappa"]]
format_persen(uji_pmml)

,Recall,Precision,F1,Kappa
Skenario,,,,
"Model SMOTE KNIME, data teks (seperti di workflow)",16.5%,12.6%,14.3%,0.011
"Model SMOTE KNIME, data di-encode",72.4%,60.7%,66.0%,0.611


**Temuan:**

- Saat diberi **data teks**, model SMOTE dari KNIME menghasilkan Recall sekitar **16,5%** dan Precision sekitar **12,6%**, hampir sama dengan hasil SMOTE di laporan Pekan 3 (Recall 16,19%, Precision 12,78%). Ini mereproduksi perilaku KNIME.
- Precision 12,6% hampir sama dengan proporsi kelas `yes` di data (11,7%), dan Kappa mendekati 0. Artinya model praktis **menebak acak**, karena lima dari tujuh kolomnya tidak terbaca.
- Saat data di-**encode** dengan benar, model yang sama langsung bekerja jauh lebih baik.

**Kesimpulan:** hasil SMOTE yang buruk di KNIME **bukan karena teknik SMOTE-nya**, melainkan karena data uji tidak ikut melewati *Category to Number*.

**Cara memperbaiki workflow KNIME:** tambahkan node **Category to Number (Apply)**. Hubungkan port PMML (output kedua) dari *Category to Number (#4)* dan data uji dari *Table Partitioner* ke node tersebut, lalu sambungkan output-nya ke *Decision Tree Predictor (#11)*.

```
Table Partitioner ── data latih → Category to Number (#4) → SMOTE → Decision Tree Learner (#8) ─┐
                 │                        └─ PMML ─┐                                          │
                 └── data uji ──────→ Category to Number (Apply) ──────→ Decision Tree Predictor (#11)
```

Hasil SMOTE ala KNIME di Colab (skenario C di bagian 4) menunjukkan kira-kira hasil yang seharusnya didapat setelah workflow diperbaiki.

Simulasi bug ini juga bisa dilakukan dengan model scikit-learn: kolom kategorikal di data uji diganti angka yang sangat besar, sehingga semua perbandingan `<=` bernilai salah, sama seperti perilaku KNIME di atas.

In [11]:
X_bug = info_7["X_uji_angka"].copy()
X_bug[:, :info_7["n_kat"]] = 1e9      # kolom kategorikal (hasil OrdinalEncoder ada di depan)

simulasi = pd.DataFrame([
    evaluasi(info_7["y_uji"], info_7["model"].predict(info_7["X_uji_angka"]), "SMOTE, data uji di-encode (benar)", "-"),
    evaluasi(info_7["y_uji"], info_7["model"].predict(X_bug), "SMOTE, data uji tidak di-encode (bug)", "-"),
]).set_index("Skenario")[["Recall", "Precision", "F1", "Accuracy", "Kappa"]]
format_persen(simulasi)

,Recall,Precision,F1,Accuracy,Kappa
Skenario,,,,,
"SMOTE, data uji di-encode (benar)",18.4%,27.9%,22.2%,84.9%,0.142
"SMOTE, data uji tidak di-encode (bug)",14.1%,15.9%,14.9%,81.2%,0.044


## 6. Eksperimen 2: 15 Fitur (semua kecuali `duration`)

Sama seperti di notebook Orange, eksperimen diulang dengan semua fitur kecuali `duration`.

In [12]:
FITUR_15 = [k for k in df.columns if k not in ("y", "duration")]
hasil_15, _ = jalankan_skenario(df, FITUR_15)
format_persen(hasil_15)

Data latih: 31647 baris, data uji: 13564 baris


,Data latih,TP,FP,FN,TN,Recall,Precision,F1,Accuracy,Kappa
Skenario,,,,,,,,,,
A. Tanpa balancing,31647,451,888,1136,11089,28.4%,33.7%,30.8%,85.1%,0.225
B. Equal Size Sampling,7404,928,3768,659,8209,58.5%,19.8%,29.5%,67.4%,0.146
C. SMOTE ala KNIME,55890,464,903,1123,11074,29.2%,33.9%,31.4%,85.1%,0.231
D. SMOTE-NC,55890,661,1719,926,10258,41.7%,27.8%,33.3%,80.5%,0.224


In [13]:
ringkasan = pd.concat({"7 fitur": hasil_7[["Recall", "Precision", "F1"]],
                       "15 fitur": hasil_15[["Recall", "Precision", "F1"]]}, axis=1)
format_persen(ringkasan)

7 fitur                  15 fitur                 
                        Recall Precision     F1   Recall Precision     F1
Skenario                                                                 
A. Tanpa balancing       21.0%     31.5%  25.2%    28.4%     33.7%  30.8%
B. Equal Size Sampling   56.0%     17.2%  26.3%    58.5%     19.8%  29.5%
C. SMOTE ala KNIME       18.4%     27.9%  22.2%    29.2%     33.9%  31.4%
D. SMOTE-NC              40.6%     20.6%  27.4%    41.7%     27.8%  33.3%

## 7. Kesimpulan

1. **KNIME tidak bisa dijalankan di Colab, tetapi workflow-nya bisa ditulis ulang sebagai kode.** Setiap node punya padanan di pandas, scikit-learn, dan imbalanced-learn. Hasilnya tidak identik karena generator acak dan algoritma Decision Tree-nya berbeda, tetapi pola tanpa balancing dan Equal Size Sampling sama dengan KNIME.
2. **Model KNIME bisa dibawa ke Colab lewat PMML.** Model Decision Tree dari KNIME bisa dijalankan di Python tanpa KNIME. Cara ini juga berguna untuk memeriksa workflow: dari sini ditemukan bahwa data uji di cabang SMOTE tidak di-encode.
3. **Hasil SMOTE yang buruk di laporan Pekan 3 disebabkan kesalahan workflow, bukan teknik SMOTE.** Setelah data uji di-encode dengan benar, SMOTE tidak lagi menebak acak: hasilnya setara dengan tanpa balancing (sedikit di bawah dengan 7 fitur, sedikit di atas dengan 15 fitur). Workflow KNIME perlu diperbaiki dengan node *Category to Number (Apply)*, lalu dijalankan ulang untuk mendapatkan angka yang benar.
4. **SMOTE-NC lebih tepat daripada SMOTE biasa untuk data campuran.** SMOTE biasa memperlakukan kode kategori (0, 1, 2, ...) sebagai angka sungguhan. SMOTE-NC menangani kolom kategorikal secara terpisah, dan di sini memberi recall tertinggi di antara teknik oversampling (sekitar 41%).
5. **Efek balancing bergantung pada kualitas fitur**, sama dengan temuan di notebook Orange: dengan 15 fitur, semua skenario membaik.

### Cara Bertukar Data dan Model antara KNIME dan Colab

| Kebutuhan | Dari KNIME ke Colab | Dari Colab ke KNIME |
|---|---|---|
| Data | *CSV Writer* → upload ke GitHub/Drive → `pd.read_csv` | `df.to_csv()` → *CSV Reader* |
| Model | *PMML Writer* → dibaca di Python (seperti bagian 5) | Model scikit-learn diekspor ke PMML (misalnya dengan `sklearn2pmml`) → *PMML Reader* + *PMML Predictor* |
| Kode | – | Kode Python dari notebook bisa dipakai di node *Python Script* KNIME (butuh ekstensi KNIME Python Integration) |